In [1]:
# load the checked output CSV file
import pandas as pd

# Load the checked output CSV file
checked_df = pd.read_csv('Data/output_checked.csv')
checked_df

,name,area,postcode,latitude,longitude
0,Barking,Barking and Dagenham,IG11 0BB,51.530000,0.088874
1,Dagenham,Barking and Dagenham,RM10 7ES,51.559469,0.157028
2,Dowgate,City,EC4R 3UE,51.510035,-0.090101
3,Shoreditch,Hackney,EC1V 9EY,51.526622,-0.085486
4,Stoke Newington,Hackney,N16 0AR,51.562572,-0.076866
...,...,...,...,...,...
98,Wandsworth,Wandsworth,SW18 1RL,51.456383,-0.201401
99,Battersea,Wandsworth,SW11 2TL,51.467115,-0.169294
100,Tooting,Wandsworth,SW17 7SQ,51.437827,-0.162705
101,Paddington,Westminster,W2 6NL,51.520072,-0.183319


In [4]:
# for the rows with missing geocodes, try to geocode them again using geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
geolocator = Nominatim(user_agent="fire_station_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
for index, row in checked_df.iterrows():
    if pd.isnull(row['latitude']) or pd.isnull(row['longitude']):
        location = geocode(f"{row['name']}, London")
        print(f"Geocoding {row['name']}: {location}")
        if location:
            checked_df.at[index, 'latitude'] = location.latitude
            checked_df.at[index, 'longitude'] = location.longitude
checked_df

Geocoding Enfield: Enfield, Greater London, England, EN2 6LD, United Kingdom
Geocoding Bexley: Bexley, London Borough of Bexley, Greater London, England, DA5 1LD, United Kingdom
Geocoding Eltham: Eltham, Greater London, England, SE9 5AS, United Kingdom
Geocoding Lee Green: Lee Green, Poverest, St Mary Cray, London Borough of Bromley, Greater London, England, BR5 2DL, United Kingdom
Geocoding Plumstead: Plumstead, Royal Borough of Greenwich, Greater London, England, SE18 2HH, United Kingdom
Geocoding Old Kent Road: Old Kent Road, Astley Estate, Old Kent Road, London Borough of Southwark, Greater London, England, SE1 5LU, United Kingdom
Geocoding Hammersmith: Hammersmith, London Borough of Hammersmith and Fulham, Greater London, England, W6 9YD, United Kingdom


,name,area,postcode,latitude,longitude
0,Barking,Barking and Dagenham,IG11 0BB,51.530000,0.088874
1,Dagenham,Barking and Dagenham,RM10 7ES,51.559469,0.157028
2,Dowgate,City,EC4R 3UE,51.510035,-0.090101
3,Shoreditch,Hackney,EC1V 9EY,51.526622,-0.085486
4,Stoke Newington,Hackney,N16 0AR,51.562572,-0.076866
...,...,...,...,...,...
98,Wandsworth,Wandsworth,SW18 1RL,51.456383,-0.201401
99,Battersea,Wandsworth,SW11 2TL,51.467115,-0.169294
100,Tooting,Wandsworth,SW17 7SQ,51.437827,-0.162705
101,Paddington,Westminster,W2 6NL,51.520072,-0.183319


In [6]:
# Convert to British National Grid (EPSG:27700)
import pyproj
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)
checked_df['easting'], checked_df['northing'] = transformer.transform(checked_df['longitude'].values, checked_df['latitude'].values)
checked_df

,name,area,postcode,latitude,longitude,easting,northing
0,Barking,Barking and Dagenham,IG11 0BB,51.530000,0.088874,544992.902885,183300.180862
1,Dagenham,Barking and Dagenham,RM10 7ES,51.559469,0.157028,549623.356235,186714.188598
2,Dowgate,City,EC4R 3UE,51.510035,-0.090101,532637.238240,180740.710344
3,Shoreditch,Hackney,EC1V 9EY,51.526622,-0.085486,532909.219426,182593.568436
4,Stoke Newington,Hackney,N16 0AR,51.562572,-0.076866,533402.031897,186607.063610
...,...,...,...,...,...,...,...
98,Wandsworth,Wandsworth,SW18 1RL,51.456383,-0.201401,525060.558867,174578.738179
99,Battersea,Wandsworth,SW11 2TL,51.467115,-0.169294,527261.298771,175827.380140
100,Tooting,Wandsworth,SW17 7SQ,51.437827,-0.162705,527800.651014,172582.007009
101,Paddington,Westminster,W2 6NL,51.520072,-0.183319,526141.005267,181692.215639


In [7]:
# save the updated DataFrame back to CSV
checked_df.to_csv('Data/output_final.csv', index=False)